# Mémoire & GPU · **côté Python**

*Piste Python / Colab — analogue de `4_memoire_et_gpu.jl`.*

Localité (numpy est **row-major**, l'inverse de Julia), intensité arithmétique & **batching**, puis
le **GPU avec PyTorch**.

> Pour le GPU : Colab ▸ *Modifier ▸ Paramètres du notebook ▸ GPU*.

In [ ]:
import timeit

def best_time(fn, repeat=7):
    """Temps par appel (s), façon %timeit : auto-échelle puis min sur `repeat` essais."""
    number = 1
    while timeit.timeit(fn, number=number) < 0.05:
        number *= 10
    return min(timeit.repeat(fn, number=number, repeat=repeat)) / number

def ms(t): return f"{t*1e3:.3f} ms"
def us(t): return f"{t*1e6:.2f} µs"


In [ ]:
import numpy as np

## 1. Localité : numpy est *row-major* (C-order)

Deux éléments d'une même **ligne** sont voisins en mémoire (C-order) — **l'inverse de Julia**
(column-major). On somme la même matrice selon le même axe, mais rangée en C puis en Fortran :
seul le **layout** change.

In [ ]:
N = 8000
A = np.random.rand(N, N)            # C-order (lignes contiguës) par défaut
A_f = np.asfortranarray(A)          # mêmes nombres, colonnes contiguës

# réduction le long de l'axe 0 (on descend les colonnes) :
t_c = best_time(lambda: A.sum(axis=0))     # C-order : on saute d'une ligne à l'autre
t_f = best_time(lambda: A_f.sum(axis=0))   # F-order : colonnes contiguës → localité
print("somme axis=0, C-order :", ms(t_c))
print("somme axis=0, F-order :", ms(t_f))
print(f"ratio : {t_c/t_f:.2f}×")

La réduction est plus rapide quand l'axe parcouru est **contigu** en mémoire. En numpy, l'axe
contigu par défaut est le **dernier** (lignes) ; en Julia, c'est le **premier** (colonnes). Se tromper
de sens ne donne pas un résultat faux — juste un code plus lent.

## 2. Intensité arithmétique : matrice-vecteur vs matrice-matrice

In [ ]:
n = 2000
M  = np.random.rand(n, n)
vv = np.random.rand(n)
M2 = np.random.rand(n, n)

print("matrice × vecteur :", ms(best_time(lambda: M @ vv)))   # ~2n² ops / n² données → memory-bound
print("matrice × matrice :", ms(best_time(lambda: M @ M2)))   # ~2n³ ops / n² données → compute-bound

- **mat × vecteur** : ~2 ops par donnée chargée → on attend la mémoire (*memory-bound*).
- **mat × matrice** : ~n ops par donnée → on sature le calcul (*compute-bound*, BLAS niveau 3).

## 3. Batching : le débit **par exemple** (numpy)

Une « couche » `W` (n→n) appliquée à un batch de `B` exemples empilés en colonnes (`W @ X`).

In [ ]:
W = np.random.rand(n, n).astype(np.float32)
print("batch B | temps total (ms) | par exemple (µs) | exemples/s")
print("-"*60)
for B in (1, 8, 64, 256, 1024):
    X = np.random.rand(n, B).astype(np.float32)
    t = best_time(lambda: W @ X)
    print(f"{B:6d}  | {t*1e3:12.2f} | {t/B*1e6:14.2f} | {B/t:12.0f}")

Le temps **total** monte avec `B`, mais le temps **par exemple** *chute* : `W` chargé une fois sert
à tout le batch. C'est « matrice-vecteur (memory-bound) → matrice-matrice (compute-bound) ».

## 4. GPU avec PyTorch

`torch` choisit CPU ou CUDA selon le *device* des tenseurs — comme le multiple dispatch de Julia
choisit la version GPU pour des `CuArray`. Mêmes opérations, matériel différent.

In [ ]:
import torch
gpu = torch.cuda.is_available()
print("CUDA dispo :", gpu, "|", torch.cuda.get_device_name(0) if gpu else "CPU seulement")

def bench_torch(fn, gpu, repeat=5):
    if gpu: torch.cuda.synchronize()
    # warm-up
    fn();  torch.cuda.synchronize() if gpu else None
    import time
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter(); fn()
        if gpu: torch.cuda.synchronize()
        best = min(best, time.perf_counter() - t0)
    return best

In [ ]:
# Produit matriciel CPU vs GPU
M = 4096
Ac = torch.rand(M, M); Bc = torch.rand(M, M)
print("matmul CPU :", ms(bench_torch(lambda: Ac @ Bc, gpu=False)))
if gpu:
    Ag = Ac.cuda(); Bg = Bc.cuda()
    print("matmul GPU :", ms(bench_torch(lambda: Ag @ Bg, gpu=True)))

In [ ]:
# Batching CPU vs GPU : le débit par exemple
if gpu:
    Wt = torch.rand(n, n, device="cuda")
    print("batch B | CPU µs/ex | GPU µs/ex | accélération")
    print("-"*48)
    Wc = Wt.cpu()
    for B in (1, 8, 64, 256, 1024):
        Xc = torch.rand(n, B); Xg = Xc.cuda()
        tc = bench_torch(lambda: Wc @ Xc, gpu=False)
        tg = bench_torch(lambda: Wt @ Xg, gpu=True)
        print(f"{B:6d}  | {tc/B*1e6:9.2f} | {tg/B*1e6:9.2f} | {tc/tg:10.1f}×")
else:
    print("(pas de GPU : activez l'accélérateur GPU dans Colab)")

In [ ]:
# Exemple conducteur : Monte-Carlo de π sur GPU
def pi_torch(n, device):
    x = torch.rand(n, device=device); y = torch.rand(n, device=device)
    return 4.0 * torch.count_nonzero(x*x + y*y <= 1.0).item() / n

n_mc = 100_000_000
print("π (réf) :", np.pi)
print("π torch CPU :", pi_torch(n_mc, "cpu"))
if gpu:
    print("π torch GPU :", pi_torch(n_mc, "cuda"))
    print("CPU :", ms(bench_torch(lambda: pi_torch(n_mc, "cpu"), gpu=False)))
    print("GPU :", ms(bench_torch(lambda: pi_torch(n_mc, "cuda"), gpu=True)))

## Bilan — à comparer avec Julia/CUDA

| Idée | Python | Julia |
|---|---|---|
| ordre mémoire | **row-major** (C) | **column-major** |
| batching | `W @ X` numpy/torch | `W * X` |
| GPU | PyTorch (`.cuda()`) | CUDA.jl (`cu(...)`, dispatch) |

> Même histoire : respecter la **localité**, **batcher** pour passer de memory-bound à compute-bound,
> et **alimenter** le GPU (rentable seulement à grande échelle). Comparez le facteur GPU/CPU obtenu ici
> à celui du notebook Julia.